In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

# Check if TensorFlow was built with CUDA
print("Built with CUDA:", tf.test.is_built_with_cuda())

# List available GPUs
gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", gpus)


# Stable Diffusion (PyTorch) – End-to-End Pipeline

This notebook:
- Uses **PyTorch only**
- Loads **Stable Diffusion v1.5**
- Generates images from text prompts
- Evaluates outputs using practical diffusion metrics
- Saves the **entire pipeline + weights**
- Allows reuse of the saved model in a **VS Code frontend (`main.py`)**

Frameworks:
- diffusers
- transformers
- torch
- safetensors


In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision safetensors ftfy pillow


## Import Libraries and Configure Device

We detect the best available device:
- CUDA (GPU)
- Apple MPS (Mac)
- CPU fallback


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
import numpy as np
import os


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)


## Load Stable Diffusion v1.5 (PyTorch)

- Loads pretrained weights from Hugging Face
- Uses Euler Ancestral scheduler for better quality
- Enables memory optimizations


In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,   # disable NSFW checker (optional)
)

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipe.scheduler.config
)

pipe = pipe.to(device)
pipe.enable_attention_slicing()

print("Pipeline loaded successfully")


## Text Prompt Definition

- `prompt` describes what we want
- `negative_prompt` tells the model what to avoid


In [ ]:
prompt = "ultra-detailed portrait of a red fox wearing a tiny scarf, cinematic lighting, 35mm"
negative_prompt = "blurry, lowres, jpeg artifacts, extra fingers, text, watermark"


## Prompt Enhancement (User → Professional Prompt)

Users usually give **short, vague prompts**.
Diffusion models perform best with:
- Subject clarity
- Style descriptors
- Camera & lighting
- Quality boosters

This cell:
- Takes a raw user prompt
- Enhances it into a **high-quality, professional diffusion prompt**
- Keeps it deterministic and controllable


In [ ]:
def enhance_prompt(
    user_prompt: str,
    style: str = "photorealistic",
    mood: str = "cinematic",
    camera: str = "35mm lens, shallow depth of field",
    lighting: str = "soft cinematic lighting, global illumination",
    quality: str = "ultra-detailed, 8k, high sharpness, professional color grading"
):
    """
    Enhances a short user prompt into a professional Stable Diffusion prompt.
    """

    enhanced_prompt = f"""
    {user_prompt},
    {style},
    {mood},
    {lighting},
    {camera},
    {quality},
    highly detailed textures,
    realistic anatomy,
    award-winning photography,
    masterpiece
    """

    # Clean formatting
    enhanced_prompt = " ".join(enhanced_prompt.split())
    return enhanced_prompt



## Example: Enhance User Prompt



In [ ]:
# Raw prompt from user (UI / frontend / input box)
user_prompt = "a red fox wearing a red scarf"

# Enhance it
enhanced_prompt = enhance_prompt(
    user_prompt,
    style="hyper-realistic animal portrait",
    mood="cinematic fantasy",
    camera="85mm DSLR lens, f1.8",
    lighting="dramatic rim lighting, soft shadows"
)

negative_prompt = (
    "blurry, lowres, jpeg artifacts, extra limbs, "
    "bad anatomy, text, watermark, logo"
)

print("USER PROMPT:\n", user_prompt)
print("\nENHANCED PROMPT:\n", enhanced_prompt)


## Image Generation

Important parameters:
- `num_inference_steps`: image quality vs speed
- `guidance_scale`: prompt adherence
- `seed`: reproducibility


In [ ]:
generator = torch.Generator(device=device).manual_seed(42)

image = pipe(
    prompt=enhanced_prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    height=512,
    width=512,
    generator=generator
).images[0]

image.save("generated_image_enhanced.png")
image


## Evaluation Metrics for Diffusion Models

Diffusion models don't use accuracy.
Common evaluation approaches:
1. **CLIP similarity score** (text–image alignment)
2. **Inference latency**
3. **Determinism (seed reproducibility)**

Below we compute CLIP similarity using OpenAI CLIP.


In [ ]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

inputs = clip_processor(
    text=[prompt],
    images=image,
    return_tensors="pt",
    padding=True
).to(device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    clip_score = outputs.logits_per_image.item()

print("CLIP similarity score:", clip_score)


## Save Model and Weights (For VS Code / Frontend)

This saves:
- UNet
- VAE
- Text Encoder
- Scheduler
- Tokenizer

You can zip and download this folder.


In [ ]:
SAVE_DIR = "/content/stable_diffusion_saved"

pipe.save_pretrained(SAVE_DIR)
print(f"Model saved to: {SAVE_DIR}")


In [ ]:
import os
print(os.listdir("/content"))


In [ ]:

!mv /content/stable_diffusion_saved.zip /content/drive/MyDrive/
